# Evaluasi Model Adaptive LoRa — (MedGemma Chest X-Ray)

## Prepare GPU

In [ ]:
"""
AUTO-SELECT GPU
===============
Memindai semua GPU yang tersedia via nvidia-smi, lalu memilih satu GPU
dengan VRAM bebas terbanyak dan utilisasi terendah.
CUDA_VISIBLE_DEVICES di-set SEBELUM torch di-import agar efektif.
"""

import os
import subprocess
import sys

N_GPUS_TO_USE = 1   # ← ubah sesuai kebutuhan

QUERY_FIELDS = "index,name,memory.free,memory.total,utilization.gpu"
try:
    result = subprocess.run(
        ["nvidia-smi", f"--query-gpu={QUERY_FIELDS}", "--format=csv,noheader,nounits"],
        capture_output=True, text=True, check=True,
    )
except FileNotFoundError:
    sys.exit("ERROR: nvidia-smi tidak ditemukan. Pastikan driver NVIDIA terinstall.")
except subprocess.CalledProcessError as e:
    sys.exit(f"ERROR: nvidia-smi gagal: {e.stderr.strip()}")

gpu_info = []
for line in result.stdout.strip().splitlines():
    parts = [p.strip() for p in line.split(",")]
    if len(parts) < 5:
        continue
    try:
        gpu_info.append({
            "index"    : int(parts[0]),
            "name"     : parts[1],
            "mem_free" : int(parts[2]),
            "mem_total": int(parts[3]),
            "util_gpu" : int(parts[4]),
        })
    except ValueError:
        continue

if not gpu_info:
    sys.exit("ERROR: Tidak ada GPU yang terdeteksi dari nvidia-smi.")

header = f"{'IDX':>3}  {'Name':<30}  {'Free MiB':>10}  {'Total MiB':>10}  {'Util%':>6}"
sep    = "=" * len(header)
print(sep)
print("  GPU INFO (nvidia-smi)")
print(sep)
print(f"  {header}")
print("-" * len(header))
for g in gpu_info:
    marker = " ◄" if g is sorted(gpu_info, key=lambda x: (-x["mem_free"], x["util_gpu"]))[0] else ""
    print(
        f"  {g['index']:>3}  {g['name']:<30}  "
        f"{g['mem_free']:>10,}  {g['mem_total']:>10,}  {g['util_gpu']:>5}%{marker}"
    )
print(sep)

sorted_gpus  = sorted(gpu_info, key=lambda g: (-g["mem_free"], g["util_gpu"]))
selected     = sorted_gpus[:N_GPUS_TO_USE]
selected_ids = [str(g["index"]) for g in selected]

# *** WAJIB di-set SEBELUM import torch ***
os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(selected_ids)

print(f"\nGPU terpilih (N_GPUS_TO_USE={N_GPUS_TO_USE}):")
for rank, g in enumerate(selected):
    print(
        f"  cuda:{rank} ← GPU {g['index']}  {g['name']}"
        f"  |  VRAM bebas: {g['mem_free']:,} MiB / {g['mem_total']:,} MiB"
        f"  |  Util: {g['util_gpu']}%"
    )
print(f"\nCUDA_VISIBLE_DEVICES = \"{os.environ['CUDA_VISIBLE_DEVICES']}\"")
print("OK — lanjutkan ke cell berikutnya.")

## 1. Konfigurasi & Persiapan Data

Konfigurasi path, dekripsi dataset, dan validasi adapter.

In [ ]:
import io
import getpass
import torch
import pandas as pd
from pathlib import Path
from cryptography.fernet import Fernet, InvalidToken

# ============================================================
# KONFIGURASI PATH — sesuaikan dengan lokasi file Anda
# ============================================================
ENCRYPTED_CSV_PATH  = Path("SIAP_TRAINING.csv.encrypted")
ENCRYPTED_IMAGE_DIR = Path("./IMAGE/GAMBAR_TERBARU_ENKRIPSI")

# Path adapter hasil training AdaLoRA v3
ADAPTER_DIR = "./LORA_ADAPTER_MEDGEMMA_4B_IT"

# Random seed untuk reproducibility split
RANDOM_STATE = 42

# ============================================================
# INPUT ENCRYPTION KEY (Fernet) — dimasukkan saat runtime
# ============================================================
raw_key = getpass.getpass("Masukkan Fernet encryption key (input tersembunyi): ").strip()
if not raw_key:
    raise ValueError("Encryption key tidak boleh kosong.")

try:
    fernet = Fernet(raw_key.encode() if isinstance(raw_key, str) else raw_key)
    fernet.decrypt(fernet.encrypt(b"key_validation_test"))
    print("Encryption key valid.")
except InvalidToken:
    raise ValueError("Encryption key tidak valid — pastikan format Fernet yang benar.")
except Exception as e:
    raise ValueError(f"Error validasi key: {e}")

# ============================================================
# VALIDASI PATH
# ============================================================
assert ENCRYPTED_CSV_PATH.exists(),  f"CSV tidak ditemukan: {ENCRYPTED_CSV_PATH}"
assert ENCRYPTED_IMAGE_DIR.exists(), f"Folder gambar tidak ditemukan: {ENCRYPTED_IMAGE_DIR}"
assert Path(ADAPTER_DIR).exists(),   f"Adapter tidak ditemukan: {ADAPTER_DIR}"

# ============================================================
# DEKRIPSI CSV DI MEMORI
# ============================================================
with open(ENCRYPTED_CSV_PATH, "rb") as f:
    encrypted_csv_bytes = f.read()

try:
    decrypted_csv_bytes = fernet.decrypt(encrypted_csv_bytes)
except InvalidToken:
    raise ValueError("Gagal dekripsi CSV — periksa encryption key.")

df = pd.read_csv(io.BytesIO(decrypted_csv_bytes))

REQUIRED_COLS = ["image_file", "findings", "conclusion", "exam_type"]
missing = [c for c in REQUIRED_COLS if c not in df.columns]
assert not missing, f"Kolom CSV tidak lengkap, kurang: {missing}"

df = df.dropna(subset=REQUIRED_COLS)
df["findings"]   = df["findings"].astype(str)
df["conclusion"] = df["conclusion"].astype(str)
df["exam_type"]  = df["exam_type"].astype(str)
df = df.reset_index(drop=True)

print(f"Total data  : {len(df)} baris")
print(f"Adapter dir : {ADAPTER_DIR}")
print(f"\nDistribusi exam_type:\n{df['exam_type'].value_counts()}")


## 2. Reproduksi Test Split

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=RANDOM_STATE, stratify=df["exam_type"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=RANDOM_STATE, stratify=temp_df["exam_type"]
)

print(f"Train : {len(train_df)} sampel")
print(f"Val   : {len(val_df)} sampel")
print(f"Test  : {len(test_df)} sampel  ← digunakan untuk evaluasi")
print(f"\nDistribusi Test:\n{test_df['exam_type'].value_counts()}")

# Reset index agar iterasi bersih
test_df = test_df.reset_index(drop=True)


## 3. Load Model + Adaptive LoRA Adapter

In [ ]:
from transformers import AutoProcessor, AutoModelForCausalLM
from peft import PeftConfig, PeftModel

# ============================================================
# BACA KONFIGURASI ADAPTER UNTUK MENDAPAT BASE MODEL ID
# ============================================================
print(f"Membaca konfigurasi adapter dari: {ADAPTER_DIR}")
peft_config    = PeftConfig.from_pretrained(ADAPTER_DIR)
BASE_MODEL_ID  = peft_config.base_model_name_or_path
print(f"Base model    : {BASE_MODEL_ID}")

# ============================================================
# LOAD PROCESSOR
# ============================================================
# Gunakan processor yang disimpan bersama adapter agar
# konfigurasi tokenizer dan image processor identik dengan saat training
processor = AutoProcessor.from_pretrained(ADAPTER_DIR)
print("Processor loaded dari adapter dir.")

# ============================================================
# LOAD BASE MODEL
# ============================================================
n_gpu      = torch.cuda.device_count()
max_memory = {i: "170GiB" for i in range(n_gpu)}
max_memory["cpu"] = "64GB"

print(f"\nMemuat base model {BASE_MODEL_ID} dalam bfloat16...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    max_memory=max_memory,
    trust_remote_code=True,
)

# ============================================================
# TEMPELKAN ADALORA ADAPTER KE BASE MODEL
# ============================================================
print(f"Menempelkan AdaLoRA adapter dari {ADAPTER_DIR}...")
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

# Ringkasan memori setelah load
print()
for i in range(n_gpu):
    alloc  = torch.cuda.memory_allocated(i) / 1e9
    total  = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  GPU {i}: {alloc:.1f} / {total:.1f} GB")

print("\nModel + AdaLoRA Adapter siap untuk evaluasi.")

## 4. Fungsi Helper

In [ ]:
import io
from PIL import Image
from cryptography.fernet import Fernet, InvalidToken

# ============================================================
# FUNGSI HELPER — IDENTIK DENGAN SAAT TRAINING
# ============================================================

SYSTEM_PROMPT = [
    "Anda adalah dokter radiologi. Buat laporan thorax lengkap dalam bahasa Indonesia.",
    "Tolong analisis X-ray dada ini dan tuliskan temuan radiologis beserta kesimpulannya.",
    "Sebagai spesialis radiologi, apa diagnosis Anda untuk gambar rontgen berikut?",
    "Berikan evaluasi klinis dari citra radiografi thorax ini secara terperinci.",
    "Tuliskan laporan medis dari foto rontgen dada ini, mencakup temuan dan konklusi."
]

def build_inference_prompt(processor) -> str:
    """Format inferensi: hanya sisi user, tanpa jawaban model."""
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": SYSTEM_PROMPT},
            ],
        },
    ]
    return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def build_reference_text(exam_type: str, findings: str, conclusion: str) -> str:
    """
    Susun teks referensi (ground truth) dengan format yang SAMA
    dengan output yang diharapkan dari model saat training.
    """
    return (
        f"Exam Type:\n{exam_type.strip()}\n\n"
        f"Findings:\n{findings.strip()}\n\n"
        f"Conclusion:\n{conclusion.strip()}"
    )


def decrypt_image_to_pil(fernet: Fernet, img_path: Path) -> Image.Image:
    """Dekripsi file .enc di memori → PIL Image. Tidak menyentuh disk."""
    with open(img_path, "rb") as f:
        encrypted_bytes = f.read()
    try:
        decrypted_bytes = fernet.decrypt(encrypted_bytes)
    except InvalidToken as e:
        raise RuntimeError(f"Gagal dekripsi {img_path.name}: {e}")
    img = Image.open(io.BytesIO(decrypted_bytes)).convert("RGB")
    img = img.resize((896, 896), Image.Resampling.LANCZOS)
    return img


print("Helper functions siap.")
print(f"Jumlah data test : {len(test_df)} sampel")
print(f"\nContoh referensi :")
ex = test_df.iloc[0]
print(build_reference_text(ex["exam_type"], ex["findings"], ex["conclusion"])[:300] + "...")

## 5. Inferensi pada Test Set

In [ ]:
import torch
from tqdm.auto import tqdm

# ============================================================
# INFERENSI MODEL PADA SELURUH DATA TEST
# ============================================================
# Model menghasilkan prediksi teks untuk setiap gambar X-ray
# di test set, kemudian dibandingkan dengan ground truth
# menggunakan metrik ROUGE, BERTScore, dan BLEU.
# ============================================================

semua_prediksi  = []   # hasil generate model
semua_referensi = []   # ground truth laporan dokter

prompt_text = build_inference_prompt(processor)

print(f"Memulai inferensi pada {len(test_df)} data test...")
print("(Setiap sampel: dekripsi gambar → generate teks → decode output)\n")

model.eval()
with torch.no_grad():
    for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inferensi"):

        # 1. Dekripsi dan siapkan gambar
        img_path = ENCRYPTED_IMAGE_DIR / row["image_file"]
        image    = decrypt_image_to_pil(fernet, img_path)

        # 2. Tokenisasi prompt + gambar
        inputs   = processor(
            images=image,
            text=prompt_text,
            return_tensors="pt",
        ).to(model.device)

        input_len = inputs["input_ids"].shape[1]

        # 3. Generate prediksi
        #    do_sample=False → greedy decoding (deterministik & reproducible)
        #    max_new_tokens=512 → cukup untuk laporan thorax lengkap
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            temperature=1.0,    # tidak berpengaruh saat do_sample=False
        )

        # 4. Decode hanya token yang di-generate (potong bagian prompt)
        generated_tokens = outputs[0][input_len:]
        prediksi_teks    = processor.decode(generated_tokens, skip_special_tokens=True).strip()

        # 5. Susun teks referensi (ground truth) sesuai format training
        referensi_teks = build_reference_text(
            row["exam_type"], row["findings"], row["conclusion"]
        )

        semua_prediksi.append(prediksi_teks)
        semua_referensi.append(referensi_teks)

print(f"\nInferensi selesai — {len(semua_prediksi)} prediksi dihasilkan.")
print("\n" + "="*65)
print("CONTOH PREDIKSI vs REFERENSI (sampel pertama)")
print("="*65)
print(f"\n[PREDIKSI MODEL]\n{semua_prediksi[0][:400]}...")
print(f"\n[REFERENSI DOKTER]\n{semua_referensi[0][:400]}...")

In [ ]:
import pandas as pd

# ============================================================
# Setelah proses inferensi selesai
# ============================================================

# Membuat DataFrame dengan hasil prediksi dan referensi
data = {
    'prediction': semua_prediksi,
    'ground_truth': semua_referensi
}

df_prediksi_referensi = pd.DataFrame(data)

# Menyimpan DataFrame ke dalam file CSV
output_csv_path = 'FINAL_ADALORA.csv'
df_prediksi_referensi.to_csv(output_csv_path, index=False)

print(f"Hasil inferensi telah disimpan ke {output_csv_path}")

## 6. Evaluasi Metrik

### ROUGE

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)

tot_r1_prec = tot_r1_rec = tot_r1_f1 = 0.0
tot_r2_f1   = 0.0
tot_rl_f1   = 0.0
exact_match = 0
n_data      = len(semua_prediksi)

for pred, ref in zip(semua_prediksi, semua_referensi):
    pred_bersih = pred.strip().lower()
    ref_bersih  = ref.strip().lower()

    if pred_bersih == ref_bersih:
        exact_match += 1

    scores = scorer.score(ref_bersih, pred_bersih)

    tot_r1_prec += scores["rouge1"].precision
    tot_r1_rec  += scores["rouge1"].recall
    tot_r1_f1   += scores["rouge1"].fmeasure
    tot_r2_f1   += scores["rouge2"].fmeasure
    tot_rl_f1   += scores["rougeL"].fmeasure

avg_r1_prec = (tot_r1_prec / n_data) * 100
avg_r1_rec  = (tot_r1_rec  / n_data) * 100
avg_r1_f1   = (tot_r1_f1   / n_data) * 100
avg_r2_f1   = (tot_r2_f1   / n_data) * 100
avg_rl_f1   = (tot_rl_f1   / n_data) * 100
akurasi_em  = (exact_match  / n_data) * 100

print("\n" + "="*65)
print(" 📊 HASIL EVALUASI ROUGE — AdaLoRA v3")
print("="*65)
print(f"Jumlah data test          : {n_data} sampel")
print("-"*65)
print("ROUGE-1 (Kecocokan Kosakata / N-Gram Tunggal):")
print(f"  Precision               : {avg_r1_prec:.2f}%")
print(f"  Recall                  : {avg_r1_rec:.2f}%")
print(f"  F1-Score                : {avg_r1_f1:.2f}%")
print("-"*65)
print(f"ROUGE-2 F1 (Frasa 2 kata) : {avg_r2_f1:.2f}%")
print(f"ROUGE-L F1 (Alur kalimat) : {avg_rl_f1:.2f}%")
print("="*65)

# Simpan hasil untuk ringkasan akhir
rouge_results = {
    "r1_precision": avg_r1_prec,
    "r1_recall"   : avg_r1_rec,
    "r1_f1"       : avg_r1_f1,
    "r2_f1"       : avg_r2_f1,
    "rl_f1"       : avg_rl_f1,
    "exact_match" : akurasi_em,
}

### BERTScore

In [ ]:
import numpy as np
from bert_score import score as bert_score_fn

print("Menghitung BERTScore...")
print("(Memuat model bert-base-multilingual-cased ke memori...)\n")

P, R, F1 = bert_score_fn(
    semua_prediksi,
    semua_referensi,
    lang="id",
    verbose=True,
)

mean_p   = np.mean(P.numpy())  * 100
mean_r   = np.mean(R.numpy())  * 100
mean_f1  = np.mean(F1.numpy()) * 100

print("\n" + "="*65)
print(" 🧠 HASIL EVALUASI BERTSCORE — AdaLoRA v3")
print("="*65)
print("BERTScore (Kecocokan Makna / Semantik Medis):")
print(f"  Precision               : {mean_p:.2f}%")
print(f"  Recall                  : {mean_r:.2f}%")
print(f"  F1-Score                : {mean_f1:.2f}%")
print("="*65)

# Simpan hasil untuk ringkasan akhir
bertscore_results = {
    "precision": mean_p,
    "recall"   : mean_r,
    "f1"       : mean_f1,
}

### RadGraph

#### Translate to English with Medgemma Baseline

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Menentukan ID model
model_id = "google/medgemma-4b-it"

print(f"Loading tokenizer & model: {model_id}...")

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load Model ke GPU
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

print(f"Model berhasil dimuat pada device: {model.device}")

In [ ]:
import pandas as pd
import torch
from tqdm.auto import tqdm

# 1. Tentukan path file CSV Anda
file_path = "dataset.csv" 

# Baca dataset dan ambil 10 sampel pertama untuk pengujian
print(f"Membaca dataset dari: {file_path}")
df_full = pd.read_csv(file_path)
df_sample = df_full.head(475).copy() # Mengambil 10 baris pertama

print("Menampilkan 3 sampel data asli:")
display(df_sample.head(3))

# Kolom yang akan diterjemahkan
kolom_target = ['ground_truth', 'prediction']

# 2. Fungsi terjemahan dengan Prompt yang Diperketat (Strict Formatting)
def translate_report_strict(text):
    if pd.isna(text) or str(text).strip() == "":
        return ""
    
    # PROMPT BARU: Ditambahkan instruksi ketat terkait format
    system_instruction = (
        "You are an expert medical AI assistant specialized in radiology. "
        "Translate the following Indonesian radiology report into professional and accurate English medical terminology.\n"
        "CRITICAL INSTRUCTIONS:\n"
        "- Strictly preserve the exact formatting, spacing, and paragraph structure of the original text.\n"
        "- Do NOT add bullet points, dashes, or numbered lists unless they explicitly exist in the original text.\n"
        "- If the original text is a continuous paragraph or a single line, the English translation MUST also be a continuous paragraph or single line."
    )
    
    messages = [
        {"role": "user", "content": f"{system_instruction}\n\nIndonesian Report:\n{text}\n\nProvide only the English translation:"}
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages, 
        add_generation_prompt=True, 
        return_tensors="pt",
        return_dict=True
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.1, # Tetap rendah untuk konsistensi
            do_sample=True,
            top_p=0.9
        )
        
    panjang_input = inputs["input_ids"].shape[-1]
    generated_tokens = outputs[0][panjang_input:]
    terjemahan_en = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    
    return terjemahan_en.strip()

# 3. Proses Terjemahan pada 10 Sampel untuk 2 Kolom
tqdm.pandas(desc="Translating")

for kolom in kolom_target:
    if kolom in df_sample.columns:
        print(f"\nMenerjemahkan kolom: '{kolom}' (10 sampel)...")
        # Membuat nama kolom baru untuk hasil terjemahan agar bisa dibandingkan
        nama_kolom_baru = f"{kolom}_en"
        df_sample[nama_kolom_baru] = df_sample[kolom].progress_apply(translate_report_strict)
    else:
        print(f"Peringatan: Kolom '{kolom}' tidak ditemukan di dataset.")

# 4. Tampilkan Hasil Pengujian
print("\n=== HASIL PENGUJIAN 10 DATA SAMPEL ===")
# Tampilkan kolom asli berdampingan dengan hasil terjemahan untuk inspeksi visual
kolom_tampil = []
for col in kolom_target:
    if col in df_sample.columns:
        kolom_tampil.extend([col, f"{col}_en"])

display(df_sample[kolom_tampil])

# Menyimpan hasil sampel jika diperlukan
df_sample.to_csv("HASIL-TRANSLATE-ADALORA_V1.csv", index=False)
print("Sampel berhasil disimpan ke 'sampel_terjemahan_medgemma.csv'")

#### Evaluation

In [ ]:
import pandas as pd
import torch
from radgraph import F1RadGraph 

print("Membaca file CSV...")
df_eval = pd.read_csv("HASIL-TRANSLATE-DATASET.csv")

REQUIRED_COLS = ["ground_truth_en", "prediction_en"]
missing = [c for c in REQUIRED_COLS if c not in df_eval.columns]
assert not missing, f"Kolom tidak ditemukan: {missing}"

df_eval = df_eval[REQUIRED_COLS].copy()

print(f"Total baris awal : {len(df_eval)}")

df_eval = df_eval.dropna(subset=["ground_truth_en", "prediction_en"]).copy()

df_eval["ground_truth_en"] = df_eval["ground_truth_en"].astype(str).str.strip()
df_eval["prediction_en"] = df_eval["prediction_en"].astype(str).str.strip()

df_eval = df_eval[(df_eval["ground_truth_en"] != "") & (df_eval["prediction_en"] != "")].copy()

df_eval = df_eval.reset_index(drop=True)

print(f"Total baris valid : {len(df_eval)}")

hypothesis   = df_eval["prediction_en"].tolist()    # prediksi model
ground_truth = df_eval["ground_truth_en"].tolist()  # laporan dokter asli

print("\nSedang memuat model F1RadGraph...")

device_type = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Menggunakan device: {device_type}")

scorer = F1RadGraph(
    reward_level="partial",   
    model_type="radgraph-xl", 
    device=device_type        
)

# ============================================================
# EVALUASI SELURUH LAPORAN
# ============================================================
print("--- Memulai Evaluasi ---")
try:
    outputs = scorer(hyps=hypothesis, refs=ground_truth)
    
    mean_score = outputs[0]   
    all_scores = outputs[1]   
    details = outputs[2] if len(outputs) >= 3 else None 

except Exception as e:
    raise RuntimeError(f"Proses evaluasi gagal: {e}")

print("\n" + "="*60)
print(f"  ⭐ Rata-rata RadGraph F1 Score ({len(df_eval)} Laporan): {mean_score:.4f}")
print("="*60)

df_eval["radgraph_f1"] = all_scores

print(f"\nDistribusi skor keseluruhan:")
print(f"  Mean : {df_eval['radgraph_f1'].mean():.4f}")
print(f"  Std  : {df_eval['radgraph_f1'].std():.4f}")
print(f"  Min  : {df_eval['radgraph_f1'].min():.4f}")
print(f"  Max  : {df_eval['radgraph_f1'].max():.4f}")

# ============================================================
# SIMPAN SKOR PER BARIS KE CSV
# ============================================================
output_scored_csv = "DATASET_SCORE.CSV"
# CSV yang disimpan hanya akan berisi: ground_truth_en, prediction_en, dan radgraph_f1
df_eval.to_csv(output_scored_csv, index=False, encoding="utf-8-sig")
print(f"\n✅ CSV dengan skor disimpan: {output_scored_csv}")

### CheXpert

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# 1. Muat Dataset
file_path = 'dataset.csv'
df = pd.read_csv(file_path)

# Pastikan kolom ada
required_cols = ['ground_truth', 'prediction']
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Kolom '{col}' tidak ditemukan di file CSV.")

# Pastikan data teks dibaca sebagai string dan tangani NaN
df['ground_truth'] = df['ground_truth'].fillna('').astype(str)
df['prediction'] = df['prediction'].fillna('').astype(str)

# 2. Target Penyakit
# Hapus duplikasi Bronchopneumonia
TARGET_CLASSES = [
    'Tuberkulosis',
    'Efusi Pleura',
    'Kardiomegali',
    'Kalsifikasi',
    'Fibrosis',
    'Elongasi',
    'Pneumonia',
    'Bronchopneumonia',
    'Pneumotoraks',
    'Edema Paru',
    'Contusio Paru',
    'Tumor'
]

# 3. Dictionary keyword
DICTIONARY = {
    'Tuberkulosis': ['tuberkulosis', 'tb', 'tb paru', 'tb lama', 'tb aktif'],
    'Efusi Pleura': ['efusi pleura'],
    'Kardiomegali': ['kardiomegali', 'jantung membesar'],
    'Kalsifikasi': ['kalsifikasi', 'kalsifikasi aorta', 'kalsifikasi granuloma'],
    'Fibrosis': ['fibrosis'],
    'Elongasi': ['elongasi', 'elongated aorta'],
    'Pneumonia': ['pneumonia'],
    'Bronchopneumonia': ['bronkopneumonia', 'bronchopneumonia'],
    'Pneumotoraks': ['pneumotoraks', 'pneumotorax'],
    'Edema Paru': ['edema paru', 'edema pulmonum'],
    'Contusio Paru': ['contusio paru', 'kontusio paru'],
    'Tumor': ['tumor', 'massa', 'nodul', 'lesi']
}

# 4. Fungsi ekstraksi label
def extract_labels_id(text, target_classes, dictionary):
    text = text.lower()
    labels = []

    for cls in target_classes:
        keywords = dictionary.get(cls, [])  # aman jika key tidak ada
        is_positive = 0

        for kw in keywords:
            if kw in text:
                # Deteksi negasi hingga 4 kata sebelum keyword
                negation_pattern = r"(tidak|tanpa|bukan|negatif)(?:\s+\w+){0,4}\s+" + re.escape(kw)

                if not re.search(negation_pattern, text):
                    is_positive = 1
                    break

        labels.append(is_positive)

    return labels

# 5. Proses ekstraksi ke matriks biner
print(f"Memulai ekstraksi label dari {len(df)} baris data...\n")

y_true = []
y_pred = []

for _, row in df.iterrows():
    gt_labels = extract_labels_id(row['ground_truth'], TARGET_CLASSES, DICTIONARY)
    pred_labels = extract_labels_id(row['prediction'], TARGET_CLASSES, DICTIONARY)

    y_true.append(gt_labels)
    y_pred.append(pred_labels)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# 6. Hitung metrik evaluasi per kelas
print("=== HASIL EVALUASI DIAGNOSTIK MEDGEMMA ===")
print("-" * 50)

f1_scores, sensitivities, specificities, accuracies = [], [], [], []

for i, cls in enumerate(TARGET_CLASSES):
    true_class = y_true[:, i]
    pred_class = y_pred[:, i]

    # Pastikan confusion matrix selalu 2x2
    tn, fp, fn, tp = confusion_matrix(true_class, pred_class, labels=[0, 1]).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    accuracy = accuracy_score(true_class, pred_class)
    f1 = f1_score(true_class, pred_class, zero_division=0)

    sensitivities.append(sensitivity)
    specificities.append(specificity)
    accuracies.append(accuracy)
    f1_scores.append(f1)

    print(f"Kelas Target   : {cls}")
    print(f"Total Positif  : {sum(true_class)} kasus aktual di Ground Truth")
    print(f"Akurasi        : {accuracy:.2f}")
    print(f"Sensitivitas   : {sensitivity:.2f}")
    print(f"Spesifisitas   : {specificity:.2f}")
    print(f"F1-Score       : {f1:.2f}")
    print("-" * 50)

# 7. Rata-rata keseluruhan
print("=== RATA-RATA KESELURUHAN (OVERALL AVERAGE) ===")
print(f"Rata-rata Akurasi      : {np.mean(accuracies):.2f}")
print(f"Rata-rata Sensitivitas : {np.mean(sensitivities):.2f}")
print(f"Rata-rata Spesifisitas : {np.mean(specificities):.2f}")
print(f"Rata-rata F1-Score     : {np.mean(f1_scores):.2f}")